[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/08_Evaluation_and_Runtime/Evaluation_and_Runtime_Deep_Dive.ipynb)

# 1.8 Evaluation and Runtime — Deep Dive

ONNX provides two ways to execute models: the pure-Python **ReferenceEvaluator** for debugging and the high-performance **ONNX Runtime** for production. Understanding both is essential.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Two Runtimes, Two Purposes](#section-1) | ReferenceEvaluator vs ONNX Runtime |
| 2 | [ReferenceEvaluator Basics](#section-2) | Pure Python execution for debugging |
| 3 | [Evaluating Single Nodes](#section-3) | Test individual operators |
| 4 | [Verbose Debugging Mode](#section-4) | Step-by-step execution tracing |
| 5 | [Custom Operators](#section-5) | Extending ONNX with OpRun subclasses |
| 6 | [ONNX Runtime Architecture](#section-6) | How ORT executes models |
| 7 | [Performance Comparison](#section-7) | Benchmarking both runtimes |
| 8 | [Key Takeaways](#section-8) | Summary |

In [ ]:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import time

from onnx import TensorProto
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx.reference import ReferenceEvaluator
import onnxruntime as ort

print(f'ONNX Runtime: {ort.__version__}')

<a id='section-1'></a>
## Section 1: Two Runtimes, Two Purposes

### Comparison Table

| Feature | ReferenceEvaluator | ONNX Runtime |
|---------|-------------------|---------------|
| **Language** | Pure Python (NumPy) | C++ with Python bindings |
| **Speed** | Slow (10-1000× slower) | Fast (optimized kernels) |
| **GPU support** | No | Yes (CUDA, TensorRT, ROCm) |
| **Verbose mode** | Yes (step-by-step tracing) | No |
| **Custom ops** | Easy (subclass `OpRun`) | Hard (C++ implementation) |
| **Use case** | Development, debugging | Production inference |
| **Graph optimization** | None | Extensive (fusion, layout) |
| **Install** | Comes with `onnx` | Separate `onnxruntime` package |

### When to Use Each

```
┌──────────────────────────────────────────────────────────┐
│                   Development Workflow                    │
│                                                          │
│  Build Model ──► ReferenceEvaluator ──► Debug & Verify  │
│       │              (verbose=3)           correct?      │
│       │                                     │   │       │
│       │                                    No  Yes      │
│       │                                     │   │       │
│       │                                     │   ▼       │
│       │                                     │ ONNX Runtime │
│       │                                     │ (production)  │
│       ▼                                     │              │
│    Fix bugs ◄───────────────────────────────┘              │
└──────────────────────────────────────────────────────────┘
```

<a id='section-2'></a>
## Section 2: ReferenceEvaluator Basics

The `ReferenceEvaluator` executes ONNX models node-by-node using pure Python (NumPy). It's not fast, but it's invaluable for understanding exactly what happens at each step.

In [ ]:
# Build a model
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

graph = make_graph(
    [make_node('MatMul', ['X', 'A'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'lr', [X, A, B], [Y])
model = make_model(graph)
check_model(model)

# Create evaluator from model
ref_sess = ReferenceEvaluator(model)

x = np.array([[1, 2], [3, 4]], dtype=np.float32)
a = np.array([[0.5], [-0.3]], dtype=np.float32)
b = np.array([[1.0]], dtype=np.float32)

feeds = {'X': x, 'A': a, 'B': b}
result = ref_sess.run(None, feeds)

print('ReferenceEvaluator result:')
print(f'  Type: {type(result)}')
print(f'  Output: {result[0]}')
print(f'  NumPy check: {x @ a + b}')
print(f'  Match: {np.allclose(result[0], x @ a + b)}')

<a id='section-3'></a>
## Section 3: Evaluating Single Nodes

A powerful feature: you can evaluate a **single `NodeProto`** without building a complete model. This is perfect for testing individual operators.

In [ ]:
# Test EyeLike operator
eye_node = make_node('EyeLike', ['X'], ['Y'])
eye_sess = ReferenceEvaluator(eye_node)

x = np.random.randn(4, 4).astype(np.float32)
result = eye_sess.run(None, {'X': x})
print('EyeLike(X):')
print(result[0])

# Test Softmax operator
softmax_node = make_node('Softmax', ['X'], ['Y'], axis=1)
softmax_sess = ReferenceEvaluator(softmax_node)

x = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype=np.float32)
result = softmax_sess.run(None, {'X': x})
print(f'\nSoftmax(X, axis=1):')
print(f'  Output: {result[0]}')
print(f'  Row sums: {result[0].sum(axis=1)}  (should be ~1.0)')

# Test Transpose operator
transpose_node = make_node('Transpose', ['X'], ['Y'], perm=[1, 0])
trans_sess = ReferenceEvaluator(transpose_node)

x = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
result = trans_sess.run(None, {'X': x})
print(f'\nTranspose(X, perm=[1,0]):')
print(f'  Input shape: {x.shape} → Output shape: {result[0].shape}')
print(f'  Output: {result[0]}')

<a id='section-4'></a>
## Section 4: Verbose Debugging Mode

The `verbose` parameter controls how much detail is printed during execution:

| Level | Output |
|-------|--------|
| `verbose=0` | Silent (default) |
| `verbose=1` | Final result only |
| `verbose=2` | Node execution: `MatMul(X, A) -> XA` |
| `verbose=3` | + Shapes and value ranges |
| `verbose=4` | + Actual tensor data values |

This is the **killer feature** for debugging ONNX models — no other runtime provides this level of introspection.

In [ ]:
feeds_small = {
    'X': np.array([[1, 2], [3, 4]], dtype=np.float32),
    'A': np.array([[0.5], [-0.3]], dtype=np.float32),
    'B': np.array([[1.0]], dtype=np.float32)
}

print('=== verbose=2: Node execution trace ===')
sess_v2 = ReferenceEvaluator(model, verbose=2)
_ = sess_v2.run(None, feeds_small)

print('\n=== verbose=3: + Shapes and ranges ===')
sess_v3 = ReferenceEvaluator(model, verbose=3)
_ = sess_v3.run(None, feeds_small)

<a id='section-5'></a>
## Section 5: Custom Operators

The ReferenceEvaluator supports custom operators through `OpRun` subclasses. This lets you prototype new operators in pure Python before implementing them in C++.

### How Custom Ops Work

1. Subclass `OpRun` from `onnx.reference.op_run`
2. Set `op_domain` to your custom domain
3. Implement `_run()` method with NumPy operations
4. Pass the class to `ReferenceEvaluator` via `new_ops` parameter

In [ ]:
from onnx.reference.op_run import OpRun

class AddEyeLike(OpRun):
    """Custom op: adds alpha * I to a square matrix."""
    op_domain = 'custom'

    def _run(self, X, alpha=1.0):
        assert len(X.shape) == 2 and X.shape[0] == X.shape[1], \
            f'Expected square matrix, got {X.shape}'
        result = X.copy()
        np.fill_diagonal(result, result.diagonal() + alpha)
        return (result,)

# Test the custom op standalone
x = np.ones((3, 3), dtype=np.float32)
op = AddEyeLike()
result = op._run(x, alpha=2.0)

print('AddEyeLike(ones(3,3), alpha=2.0):')
print(result[0])
print(f'Diagonal: {result[0].diagonal()}  (1 + 2 = 3, correct!)')

In [ ]:
# Use custom op in a model with ReferenceEvaluator
from onnx.helper import make_opsetid

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, None])

graph = make_graph(
    [make_node('AddEyeLike', ['X'], ['Y'], domain='custom', alpha=5.0)],
    'custom_op_model', [X], [Y])
model_custom = make_model(graph, opset_imports=[
    make_opsetid('', 14), make_opsetid('custom', 1)])

sess_custom = ReferenceEvaluator(model_custom, new_ops=[AddEyeLike])

x = np.zeros((4, 4), dtype=np.float32)
result = sess_custom.run(None, {'X': x})

print('Custom op in model: AddEyeLike(zeros(4,4), alpha=5.0)')
print(result[0])
print(f'Diagonal: {result[0].diagonal()}  (0 + 5 = 5, correct!)')

<a id='section-6'></a>
## Section 6: ONNX Runtime Architecture

ONNX Runtime (ORT) is a high-performance inference engine. Understanding its architecture helps explain why it's orders of magnitude faster.

### Execution Pipeline

```
┌──────────────────────────────────────────────────────────────┐
│                    ONNX Runtime Pipeline                     │
│                                                              │
│  .onnx file ──► Graph Loading ──► Graph Optimization ──►    │
│                                   (fusion, layout,           │
│                                    constant folding)         │
│                                        │                     │
│                                        ▼                     │
│                                  Partitioning ──►            │
│                                  (assign to EPs)             │
│                                        │                     │
│                           ┌────────────┼────────────┐        │
│                           ▼            ▼            ▼        │
│                     ┌─────────┐  ┌──────────┐ ┌─────────┐  │
│                     │   CPU   │  │   CUDA   │ │TensorRT │  │
│                     │  (MLAS) │  │ (cuDNN)  │ │ (TRT)   │  │
│                     └─────────┘  └──────────┘ └─────────┘  │
│                     Execution Providers (EPs)                │
└──────────────────────────────────────────────────────────────┘
```

### Graph Optimizations

| Optimization | Description | Speedup |
|-------------|-------------|----------|
| **Constant folding** | Pre-compute ops with constant inputs | Varies |
| **Operator fusion** | Merge MatMul+Add → Gemm, Conv+Relu → FusedConv | 10-30% |
| **Layout optimization** | Convert to hardware-optimal memory layout (NHWC) | 5-20% |
| **Shape inference** | Pre-allocate exact buffer sizes | 5-10% |
| **Memory planning** | Reuse buffers for non-overlapping tensors | 20-50% less memory |

In [ ]:
# Inspect ONNX Runtime session
ort_sess = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider'])

print('ONNX Runtime Session Info:')
print(f'  Providers: {ort_sess.get_providers()}')

print(f'\n  Inputs:')
for inp in ort_sess.get_inputs():
    print(f'    {inp.name}: type={inp.type}, shape={inp.shape}')

print(f'\n  Outputs:')
for out in ort_sess.get_outputs():
    print(f'    {out.name}: type={out.type}, shape={out.shape}')

# Run inference
result_ort = ort_sess.run(None, feeds_small)
print(f'\n  Result: {result_ort[0]}')

<a id='section-7'></a>
## Section 7: Performance Comparison

Let's benchmark both runtimes across different model sizes to quantify the performance gap.

In [ ]:
sizes = [(10, 5), (50, 25), (100, 50), (200, 100), (500, 250)]
ref_times = []
ort_times = []
speedups = []

n_runs = 50

for rows, cols in sizes:
    _X = make_tensor_value_info('X', TensorProto.FLOAT, [None, rows])
    _A = make_tensor_value_info('A', TensorProto.FLOAT, [rows, cols])
    _B = make_tensor_value_info('B', TensorProto.FLOAT, [1, cols])
    _Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, cols])

    _g = make_graph(
        [make_node('MatMul', ['X', 'A'], ['XA']),
         make_node('Add', ['XA', 'B'], ['Y'])],
        'bench', [_X, _A, _B], [_Y])
    _m = make_model(_g)

    x = np.random.randn(100, rows).astype(np.float32)
    a = np.random.randn(rows, cols).astype(np.float32)
    b = np.random.randn(1, cols).astype(np.float32)
    feeds = {'X': x, 'A': a, 'B': b}

    # ReferenceEvaluator
    ref = ReferenceEvaluator(_m)
    ref.run(None, feeds)  # warm up
    t0 = time.perf_counter()
    for _ in range(n_runs):
        ref.run(None, feeds)
    ref_t = (time.perf_counter() - t0) / n_runs * 1000

    # ONNX Runtime
    ort_s = ort.InferenceSession(
        _m.SerializeToString(), providers=['CPUExecutionProvider'])
    ort_s.run(None, feeds)  # warm up
    t0 = time.perf_counter()
    for _ in range(n_runs):
        ort_s.run(None, feeds)
    ort_t = (time.perf_counter() - t0) / n_runs * 1000

    ref_times.append(ref_t)
    ort_times.append(ort_t)
    speedups.append(ref_t / ort_t)

    print(f'  [{rows:>3d}×{cols:>3d}]  Ref={ref_t:>8.3f}ms  ORT={ort_t:>8.3f}ms  '
          f'Speedup={ref_t/ort_t:>6.1f}×')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

labels = [f'{r}×{c}' for r, c in sizes]
x_pos = np.arange(len(sizes))
w = 0.35

ax1.bar(x_pos - w/2, ref_times, w, label='ReferenceEvaluator', color='#E74C3C', alpha=0.7)
ax1.bar(x_pos + w/2, ort_times, w, label='ONNX Runtime', color='#3498DB', alpha=0.7)
ax1.set_xlabel('Matrix Dimensions', fontsize=12)
ax1.set_ylabel('Latency (ms)', fontsize=12)
ax1.set_title('Inference Latency Comparison', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels(labels)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_yscale('log')

ax2.bar(x_pos, speedups, color='#2ECC71', alpha=0.7)
ax2.set_xlabel('Matrix Dimensions', fontsize=12)
ax2.set_ylabel('ORT Speedup (×)', fontsize=12)
ax2.set_title('ONNX Runtime Speedup over ReferenceEvaluator', fontsize=13, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(labels)
for i, s in enumerate(speedups):
    ax2.text(i, s + 0.5, f'{s:.0f}×', ha='center', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Numerical Correctness Verification

Despite the performance difference, both runtimes should produce **numerically identical** results (within floating-point tolerance).

In [ ]:
x = np.random.randn(100, 50).astype(np.float32)
a = np.random.randn(50, 10).astype(np.float32)
b = np.random.randn(1, 10).astype(np.float32)
feeds = {'X': x, 'A': a, 'B': b}

ref_result = ReferenceEvaluator(model).run(None, feeds)[0]
ort_result = ort.InferenceSession(
    model.SerializeToString(), providers=['CPUExecutionProvider']
).run(None, feeds)[0]
np_result = x @ a + b

print('Numerical correctness check:')
print(f'  Ref vs NumPy: max_diff={np.abs(ref_result - np_result).max():.2e}')
print(f'  ORT vs NumPy: max_diff={np.abs(ort_result - np_result).max():.2e}')
print(f'  Ref vs ORT:   max_diff={np.abs(ref_result - ort_result).max():.2e}')
print(f'  All match: {np.allclose(ref_result, ort_result, atol=1e-6)}')

<a id='section-8'></a>
## Section 8: Key Takeaways

### Runtime Selection Guide

| Scenario | Use |
|----------|-----|
| Debugging a custom model | ReferenceEvaluator (verbose=3) |
| Testing a single operator | ReferenceEvaluator (single node) |
| Prototyping custom ops | ReferenceEvaluator + OpRun |
| Production inference | ONNX Runtime |
| GPU inference | ONNX Runtime (CUDA EP) |
| Verifying numerical correctness | Both (compare outputs) |

### Critical Points

1. **ReferenceEvaluator is for debugging** — use `verbose=3` to trace data through every node.

2. **Single-node evaluation** — test operators in isolation without building a full model.

3. **Custom ops are easy** — subclass `OpRun`, implement `_run()`, pass to evaluator.

4. **ORT is 10-100× faster** — the gap grows with model complexity due to C++ kernels and graph optimization.

5. **Always verify correctness** — run the same inputs through both runtimes and compare.

---

**Congratulations!** You've completed the ONNX with Python module. Next: [Introduction to ONNX](../../02_Introduction_to_ONNX/) for broader concepts.